# Museum PGSR mesh reconstruction
Enable a Kaggle GPU and Internet. Attach the saved Museum PGSR notebook output, then set MUSEUM_OUTPUT to the directory containing scene.json and run/. Set the scene receipt and checkpoint SHA-256 values printed by the training notebook.

The worker verifies the Museum source, GPU bundle adjustment evidence, prepared data and frozen split before loading the PGSR state. A full checkpoint is required by default. Enable partial checkpoints explicitly for an extraction labeled with the saved iteration.

Upstream PGSR uses CPU legacy Open3D TSDF. To honor strict GPU policy this notebook reuses official `render.py::render_set` and replaces only its volume with CUDA tensor TSDF integration/extraction. This is a declared backend adaptation, not bitwise upstream reproduction. CPU handles I/O and mesh serialization. CUDA-unavailable builds fail without fallback. Defaults are resource presets in arbitrary SfM units; refine only as a declared extraction variant.

In [ ]:
import os,sys,json,base64,hashlib,io,zipfile,subprocess,shutil
from pathlib import Path
os.environ['CUDA_VISIBLE_DEVICES']='0'
PROJECT=Path('/tmp/pgsr-mesh-project')
OUTPUT=Path('/kaggle/working/museum_pgsr_mesh')
MUSEUM_OUTPUT=None  # Directory containing the saved Museum PGSR result.
SCENE_RECEIPT_SHA256=''
CHECKPOINT=None  # Optional absolute path within the specified Museum result.
ALLOW_PARTIAL=False
VOXEL_SIZE=0.01  # Scene units; upstream CLI default is0.002, which needs more memory.
MAX_DEPTH=5.0  # Upstream default, arbitrary SfM units; not meters without alignment.
BLOCK_COUNT=30000
EXPECTED_CHECKPOINT_SHA256=None
configuration=list(Path('/kaggle/input').rglob('museum_pgsr_mesh_settings.json'))
assert len(configuration)<=1, 'Multiple extraction configuration datasets attached'
if configuration:
    selected=json.loads(configuration[0].read_text())
    VOXEL_SIZE=selected['voxel_size'];MAX_DEPTH=selected['max_depth'];BLOCK_COUNT=selected['block_count']
    ALLOW_PARTIAL=selected['allow_partial']
    EXPECTED_CHECKPOINT_SHA256=selected.get('checkpoint_sha256')
if MUSEUM_OUTPUT is None or not SCENE_RECEIPT_SHA256 or not EXPECTED_CHECKPOINT_SHA256:
    raise ValueError('Set the Museum result directory, scene receipt hash and checkpoint hash')
assert VOXEL_SIZE>0 and 0<MAX_DEPTH<65.535 and BLOCK_COUNT>0
assert not PROJECT.exists() and not OUTPUT.exists(), 'Use a fresh session; existing results are preserved.'
OUTPUT.mkdir(parents=True)
subprocess.run(['nvidia-smi'],check=True)

## Restore embedded project helpers and set up the pinned CUDA environment

In [ ]:
blob=base64.b64decode('...')
assert hashlib.sha256(blob).hexdigest()=='f1c7e6bfac84e842e5e98fad475ca83107f87157e97d78adca5ea201bb7fb093'
PROJECT.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    for item in z.infolist():
        p=Path(item.filename)
        assert not p.is_absolute() and '..' not in p.parts and chr(92) not in item.filename
        assert (item.external_attr>>16)&0o170000!=0o120000
    z.extractall(PROJECT)
os.chdir(PROJECT)
with (OUTPUT/'setup.log').open('w') as log:
    result=subprocess.run(['bash','scripts/kaggle_pgsr/setup_linux.sh'],stdout=log,stderr=subprocess.STDOUT)
if result.returncode:
    print((OUTPUT/'setup.log').read_text()[-12000:]);raise RuntimeError('Pinned PGSR CUDA build failed')
PY=PROJECT/'.tools/pgsr_env/bin/python'
with (OUTPUT/'open3d_setup.log').open('w') as log:
    subprocess.run([str(PY),'-m','pip','install','open3d==0.19.0'],stdout=log,stderr=subprocess.STDOUT,check=True)

## Locate checkpoint/data, render train views, and extract the mesh
Float depth/normal NPZ files and masks accompany upstream JPEG previews. All components are preserved; no largest-component-only cleaning is applied. Optimizer state stays off the GPU and is not stepped.

In [ ]:
worker="\nimport os,sys,json,gc,hashlib,importlib.util,_codecs,time\nfrom pathlib import Path\nfrom types import SimpleNamespace\nimport numpy as np\nimport torch\nimport open3d as o3d\nfrom torch.torch_version import TorchVersion\nfrom src.common.io import sha256,read_json,write_json\nfrom src.pgsr.native import initialize\nfrom src.pgsr.data import SceneAdapter\nfrom scripts.kaggle.museum_handoff import VIDEO_SHA256, validate_scene\nsettings=read_json(Path(sys.argv[1]));output=Path(settings['output'])\nenv=initialize()\nif not o3d.core.cuda.is_available():raise RuntimeError('BLOCKED_BACKEND: Open3D CUDA unavailable; no CPU fusion fallback')\ndevice=o3d.core.Device('CUDA:0')\nprobe=o3d.core.Tensor([1.],device=device)+1\nassert float(probe.cpu().numpy()[0])==2.\ninput_root=Path(settings['museum_output']);candidates=[]\nreceipt=read_json(input_root/'scene.json')\nif sha256(input_root/'scene.json')!=settings['scene_receipt_sha256']:\n    raise ValueError('Museum scene receipt checksum mismatch')\ndataset_id=receipt['dataset_id']\nif Path(dataset_id).name!=dataset_id:raise ValueError('Unsafe dataset ID')\nsource=input_root/'scene'/dataset_id\nif validate_scene(source,input_root/'metrics.json')!=receipt:\n    raise ValueError('Museum prepared scene verification failed')\nfor config_path in input_root.rglob('resolved_config.json'):\n    cfg=read_json(config_path)\n    if cfg.get('method')!='B2_PGSR':continue\n    if cfg.get('dataset')!=f'datasets/tanks_and_temples/museum/prepared/video_v1/{dataset_id}':continue\n    run=config_path.parent.parent if config_path.parent.name=='configs' else config_path.parent\n    pointer=run/'checkpoints/latest.json'\n    if pointer.exists():\n        p=read_json(pointer);cp=run/'checkpoints'/p['path']\n        if cp.parent.resolve()!=(run/'checkpoints').resolve():raise ValueError('Unsafe checkpoint pointer')\n        if cp.is_file() and p['iteration']>300:candidates.append((p['iteration'],cp,cfg,p,run))\nif settings.get('checkpoint'):\n    candidates=[x for x in candidates if str(x[1])==settings['checkpoint']]\nif not candidates:raise FileNotFoundError('Attach the saved Museum PGSR notebook output with its run and scene directories')\ncandidates.sort(key=lambda x:(-x[0],str(x[1])))\niteration,checkpoint,cfg,pointer,run=candidates[0]\nif sha256(checkpoint)!=pointer['sha256']:raise ValueError('Checkpoint SHA256 mismatch')\nif settings.get('expected_checkpoint_sha256') and pointer['sha256']!=settings['expected_checkpoint_sha256']:\n    raise ValueError('Mounted model differs from the configured verified recovery')\nif any(x[0]==iteration and x[3]['sha256']!=pointer['sha256'] for x in candidates):\n    raise ValueError('Ambiguous equally recent PGSR models: set CHECKPOINT explicitly')\nsafe=[np.core.multiarray._reconstruct,np.core.multiarray.scalar,np.ndarray,np.dtype,TorchVersion,_codecs.encode]\nsafe += [type(np.dtype(t)) for t in (np.uint32,np.int32,np.int64,np.float32,np.float64)]\ntorch.serialization.add_safe_globals(safe)\ntry:payload=torch.load(checkpoint,map_location='cpu',weights_only=True,mmap=True)\nfinally:torch.serialization.clear_safe_globals()\nif payload.get('schema')!='pgsr-full-checkpoint-v1' or payload['iteration']!=iteration:raise ValueError('Not a compatible full PGSR checkpoint')\nfrom src.common.io import fingerprint\nif payload['identity']['config']!=fingerprint(cfg) or payload['identity']['native_revision']!=env['native_revision']:\n    raise ValueError('Checkpoint configuration/native revision mismatch')\nif iteration<cfg['iterations'] and not settings['allow_partial']:raise ValueError('Partial model; ALLOW_PARTIAL must be explicit')\ndataset=Path.cwd()/cfg['dataset']\nif (payload['identity']['metadata']!=sha256(source/'metadata.json')\n        or payload['identity']['split']!=receipt['split_hash']):\n    raise ValueError('PGSR checkpoint was trained on a different scene or split')\ndataset.parent.mkdir(parents=True,exist_ok=True)\ndataset.symlink_to(source.resolve(),target_is_directory=True)\ndata=SceneAdapter(cfg,output/'qualification')\nif data.identity!={k:payload['identity'][k] for k in ('metadata','split')}:raise ValueError('Frozen data/camera split differs')\nfrom scene.gaussian_model import GaussianModel\nmodel=GaussianModel(cfg['sh_degree']);cap=payload['native_capture']\nif len(cap)!=16:raise ValueError('Native PGSR capture mismatch')\nfor name,index in (('_xyz',1),('_features_dc',3),('_features_rest',4),('_scaling',5),('_rotation',6),('_opacity',7)):\n    value=cap[index]\n    if not torch.isfinite(value).all():raise ValueError('Nonfinite model parameters')\n    setattr(model,name,torch.nn.Parameter(value.to('cuda:0'),requires_grad=False))\nmodel.active_sh_degree=cap[0];model.use_app=False\ndel cap,payload;gc.collect();torch.cuda.empty_cache()\n# Import the pinned official renderer/extraction loop, bypassing its destructive Scene reader.\nspec=importlib.util.spec_from_file_location('official_pgsr_render',Path('external/PGSR/render.py'))\nofficial=importlib.util.module_from_spec(spec);spec.loader.exec_module(official)\noriginal_render=official.render\nmaps=output/'maps';maps.mkdir()\ndef export_render(camera,*args,**kwargs):\n    package=original_render(camera,*args,**kwargs)\n    z=package['plane_depth'].squeeze();normal=package['rendered_normal']\n    norm=normal.norm(dim=0)\n    valid=torch.isfinite(z)&(z>0)&(z<settings['max_depth'])&torch.isfinite(normal).all(0)&(norm>1e-6)\n    # Same native 80-degree grazing-angle filter, plus explicit undefined-value masks.\n    ray=torch.nn.functional.normalize(camera.get_rays(),dim=-1)\n    dn=torch.nn.functional.normalize(package['depth_normal'].permute(1,2,0),dim=-1)\n    valid &= (ray*dn).sum(-1).abs()>=np.cos(np.deg2rad(80.))\n    package['plane_depth']=torch.where(valid[None],package['plane_depth'],0)\n    unit=torch.where(valid[None],normal/norm.clamp_min(1e-8)[None],0)\n    np.savez_compressed(maps/f'{camera.image_id:06d}.npz',depth=package['plane_depth'].squeeze().cpu().numpy(),\n        normal=unit.permute(1,2,0).cpu().numpy(),valid=valid.cpu().numpy(),\n        K=camera.get_k().cpu().numpy(),Tcw=camera.world_view_transform.T.cpu().numpy())\n    return package\nofficial.render=export_render\nclass CudaVolume:\n    def __init__(self):\n        self.grid=o3d.t.geometry.VoxelBlockGrid(attr_names=('tsdf','weight','color'),\n            attr_dtypes=(o3d.core.float32,)*3,attr_channels=(1,1,3),voxel_size=settings['voxel_size'],\n            block_resolution=16,block_count=settings['block_count'],device=device)\n        assert str(self.grid.attribute('tsdf').device)=='CUDA:0'\n        self.frames=0\n    def integrate(self,rgbd,intrinsic,pose):\n        # Native RGBD decoding is I/O; TSDF allocation/integration/extraction stay on CUDA.\n        depth_array=np.asarray(rgbd.depth).astype(np.float32,copy=True)\n        color_array=np.asarray(rgbd.color)\n        if color_array.dtype==np.uint8:\n            color_array=color_array.astype(np.float32)/255.\n        else:\n            color_array=color_array.astype(np.float32,copy=True)\n        depth=o3d.t.geometry.Image(o3d.core.Tensor(depth_array,device=device))\n        color=o3d.t.geometry.Image(o3d.core.Tensor(color_array,device=device))\n        if depth.as_tensor().dtype!=o3d.core.float32 or color.as_tensor().dtype!=o3d.core.float32:\n            raise TypeError('CUDA TSDF requires matching Float32 depth and color tensors')\n        k=o3d.core.Tensor(intrinsic.intrinsic_matrix,o3d.core.float64)\n        t=o3d.core.Tensor(pose,o3d.core.float64)\n        blocks=self.grid.compute_unique_block_coordinates(depth,k,t,1.,settings['max_depth'],4.)\n        self.grid.integrate(blocks,depth,color,k,k,t,1.,settings['max_depth'],4.)\n        self.frames+=1\n        if self.grid.hashmap().size()>settings['block_count']:raise RuntimeError('Declared voxel block budget exceeded')\n    def extract(self):\n        mesh=self.grid.extract_triangle_mesh(weight_threshold=1.)\n        if str(mesh.device)!='CUDA:0':raise RuntimeError('Extraction did not return a CUDA mesh')\n        mesh.compute_vertex_normals()\n        return mesh.cpu().to_legacy()\nvolume=CudaVolume()\npipe=SimpleNamespace(convert_SHs_python=False,compute_cov3D_python=False,debug=False)\nbg=torch.full((3,),float(cfg['white_background']),device='cuda:0')\nstarted=time.perf_counter()\nwith torch.no_grad():\n    official.render_set(str(output),'train',iteration,data.train,None,model,pipe,bg,\n        app_model=None,max_depth=settings['max_depth'],volume=volume,use_depth_filter=True)\n    mesh=volume.extract()\nextraction_seconds=time.perf_counter()-started\nvertices=np.asarray(mesh.vertices);triangles=np.asarray(mesh.triangles)\nif not len(vertices) or not len(triangles) or not np.isfinite(vertices).all():raise ValueError('Empty/nonfinite reconstructed mesh')\nif triangles.min()<0 or triangles.max()>=len(vertices):raise ValueError('Invalid triangle indices')\ntarget=output/'mesh.ply'\nif not o3d.io.write_triangle_mesh(str(target),mesh):raise RuntimeError('Mesh export failed')\nreopened=o3d.io.read_triangle_mesh(str(target))\nif len(reopened.triangles)!=len(triangles) or len(reopened.vertices)!=len(vertices):\n    raise ValueError('Export reopen validation failed')\n_,components,_=reopened.cluster_connected_triangles()\nreport={'status':'SUCCESS','method':'PGSR official render_set + CUDA tensor TSDF adapter',\n    'native_revision':env['native_revision'],'checkpoint_sha256':pointer['sha256'],'iteration':iteration,\n    'training_complete':iteration==cfg['iterations'],'training_executed':False,'environment':env,\n    'scene':'Museum','source_video_sha256':VIDEO_SHA256,'split_hash':receipt['split_hash'],\n    'vertices':len(vertices),'triangles':len(triangles),'connected_components':len(components),\n    'integrated_train_views':volume.frames,'render_and_extract_seconds':extraction_seconds,\n    'mesh_size_bytes':target.stat().st_size,\n    'voxel_size':settings['voxel_size'],'sdf_trunc':4*settings['voxel_size'],\n    'depth_max':settings['max_depth'],'units':'Original arbitrary monocular SfM units; no GT alignment',\n    'maps':'positive axial Z, camera-frame unit normals, invalid zeros + boolean mask; K/Tcw included',\n    'deviations':'CUDA tensor volume replaces upstream legacy CPU volume; finite/support masks added; no largest-component cleaning; selected resource voxel preset',\n    'chamfer_distance':None,'f_score':None,'geometry_evaluation':'NOT AVAILABLE: no qualified GT protocol',\n    'mesh_sha256':sha256(target),'peak_torch_reserved_bytes':torch.cuda.max_memory_reserved(),\n    'note':'Torch memory excludes Open3D allocations; no held-out RGB fitted or used for fusion'}\nwrite_json(output/'mesh_metrics.json',report)\nprint(json.dumps(report,indent=2))\n"
(PROJECT/'mesh_worker.py').write_text(worker,encoding='utf-8')

In [ ]:
settings={'output':str(OUTPUT),'museum_output':str(MUSEUM_OUTPUT),
          'scene_receipt_sha256':SCENE_RECEIPT_SHA256.strip().lower(),
          'checkpoint':CHECKPOINT,'allow_partial':ALLOW_PARTIAL,
          'voxel_size':VOXEL_SIZE,'max_depth':MAX_DEPTH,'block_count':BLOCK_COUNT,
          'expected_checkpoint_sha256':EXPECTED_CHECKPOINT_SHA256.strip().lower()}
settings_path=OUTPUT/'extraction_config.json'
settings_path.write_text(json.dumps(settings,indent=2))
with (OUTPUT/'extraction.log').open('w') as log:
    result=subprocess.run([str(PY),str(PROJECT/'mesh_worker.py'),str(settings_path)],stdout=log,stderr=subprocess.STDOUT)
if result.returncode:
    (OUTPUT/'failure.json').write_text(json.dumps({'status':'FAILED','returncode':result.returncode,'training_executed':False}))
    print((OUTPUT/'extraction.log').read_text()[-16000:])
    raise RuntimeError('Extraction failed; logs and completed maps are preserved. No CPU fallback or retraining.')
print((OUTPUT/'mesh_metrics.json').read_text())

## Package Museum mesh and provenance
The ZIP contains the mesh, depth and normal maps, logs, extraction parameters and checksums. Geometry scores require a qualified Museum reference.

In [ ]:
manifest={}
for p in OUTPUT.rglob('*'):
    if p.is_file():
        h=hashlib.sha256()
        with p.open('rb') as f:
            for block in iter(lambda:f.read(8*1024*1024),b''):h.update(block)
        manifest[p.relative_to(OUTPUT).as_posix()]=h.hexdigest()
(OUTPUT/'manifest.json').write_text(json.dumps({'files':manifest,'training_executed':False},indent=2))
size=sum(p.stat().st_size for p in OUTPUT.rglob('*') if p.is_file())
if size>8*1024**3 or shutil.disk_usage(OUTPUT.parent).free<size*1.05:
    raise RuntimeError('ZIP safety budget exceeded; files remain available individually. Use a declared coarser extraction profile in a fresh session.')
archive=Path(shutil.make_archive('/kaggle/working/museum_pgsr_mesh','zip',OUTPUT))
from IPython.display import FileLink,display
display(FileLink(str(archive)))
print('Download:',archive,'bytes:',archive.stat().st_size)